# AskMyDocs — RAG Demo
**Chat with your documents. Get answers with sources.**

This notebook runs the full pipeline end to end using the modules in `src/`:

load PDFs → chunk → embed → store in ChromaDB → retrieve → generate an answer with Groq.

The classes live in `src/`; this notebook only wires them together.

In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Locate the project root (the folder that contains `src/`) so this notebook
# runs the same whether it is launched from the repo root or a subfolder.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/mpant/personal-projects/rag-sandbox


In [2]:
# from src import process_all_pdfs

In [3]:
from dotenv import load_dotenv

from src import (
    process_all_pdfs,
    split_documents,
    EmbeddingManager,
    VectorStore,
    RAGRetriever,
    GroqLLM,
    AdvancedRAGPipeline,
    print_rag_result,
)

# Load GROQ_API_KEY from the project's .env
load_dotenv(PROJECT_ROOT / ".env")

True

## Configuration
All the knobs in one place.

In [4]:
PDF_DIR = PROJECT_ROOT / "data" / "pdf"
VECTOR_STORE_DIR = PROJECT_ROOT / "data" / "vector_store"

EMBEDDING_MODEL = "all-MiniLM-L6-v2"
LLM_MODEL = "llama-3.3-70b-versatile"

# Set to True to wipe the stored index and re-embed every PDF from scratch.
# Leave False on normal runs so it reuses the existing index instantly.
FORCE_REINDEX = False

## Build the index
Loads, chunks, embeds, and stores the PDFs — but only when nothing is indexed yet (or when `FORCE_REINDEX` is set). On later runs it reuses the persisted ChromaDB store, so this cell returns instantly.

If you flip `FORCE_REINDEX` to `True`, use **Kernel → Restart and Run All** for a clean rebuild.

In [5]:
import shutil

if FORCE_REINDEX and VECTOR_STORE_DIR.exists():
    shutil.rmtree(VECTOR_STORE_DIR)
    print("Cleared existing vector store")

embedding_manager = EmbeddingManager(model_name=EMBEDDING_MODEL)
vectorstore = VectorStore(persist_directory=str(VECTOR_STORE_DIR))

if vectorstore.collection.count() == 0:
    documents = process_all_pdfs(str(PDF_DIR))
    chunks = split_documents(documents)

    texts = [doc.page_content for doc in chunks]
    embeddings = embedding_manager.generate_embeddings(texts)
    vectorstore.add_documents(chunks, embeddings)
else:
    print(f"Using existing index with {vectorstore.collection.count()} documents")

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 7479.45it/s]


Model loaded successfully. Embedding dimension: 384
Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0
Found 5 PDF files to process

Processing: Language Models are Few Shot Learners.pdf
  ✓ Loaded 75 pages

Processing: BERT Pretraining.pdf
  ✓ Loaded 16 pages

Processing: RAG for Knowledge Intensive NLP Task.pdf
  ✓ Loaded 19 pages

Processing: Efficient Estimation of Word Rep in Vector Space.pdf
  ✓ Loaded 12 pages

Processing: Attention is all you Need.pdf
  ✓ Loaded 15 pages

Total documents loaded: 137
Split 137 documents into 597 chunks

Example chunk:
Content: Language Models are Few-Shot Learners
Tom B. Brown∗ Benjamin Mann∗ Nick Ryder∗ Melanie Subbiah∗
Jare...
Metadata: {'producer': 'pdfTeX-1.40.17', 'creator': 'LaTeX with hyperref package', 'creationdate': '2020-07-24T00:04:08+00:00', 'author': '', 'keywords': '', 'moddate': '2020-07-24T00:04:08+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.17 (TeX Live 2016) kpat

Batches: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 19/19 [00:05<00:00,  3.32it/s]


Generated embeddings with shape: (597, 384)
Adding 597 documents to vector store...
Successfully added 597 documents to vector store
Total documents in collection: 597


## Build the pipeline
Wire the retriever, the Groq LLM, and the orchestration layer together.

In [6]:
rag_retriever = RAGRetriever(vectorstore, embedding_manager)
llm = GroqLLM(model_name=LLM_MODEL)
adv_rag = AdvancedRAGPipeline(retriever=rag_retriever, llm=llm)

Initialized Groq LLM with model: llama-3.3-70b-versatile


## Ask a question
`summarize=True` adds a 2-sentence summary; each source carries the file and page its chunk came from.

In [7]:
query = "If you were building a question answering system today, which components from the Transformer, BERT, and RAG papers would you combine and why?"

result = adv_rag.query(
    question=query,
    top_k=3,
    min_score=0.1,
    summarize=True
)

print_rag_result(result)

Retrieving documents for query: 'If you were building a question answering system today, which components from the Transformer, BERT, and RAG papers would you combine and why?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.39s/it]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
QUESTION
If you were building a question answering system today, which components from the Transformer, BERT, and RAG papers would you combine and why?

ANSWER
If I were building a question answering system today, I would combine the following components from the Transformer, BERT, and RAG papers:

1. **Transformer Architecture**: I would use the Transformer architecture as the foundation for my question answering system. The Transformer's self-attention mechanism and encoder-decoder structure have proven to be highly effective in natural language processing tasks, including question answering.
2. **BERT-based Query Encoder**: I would use a BERT-based query encoder, similar to the one used in RAG, to encode the input question. BERT's pre-training objective and large-scale training data have made it a highly effective language model, and its encoder can capture the nuances of the input question.
3. **DPR-b

## Try your own
Change `query` above, or call `adv_rag.query(...)` with a new question. The full history is kept in `adv_rag.history`.

In [8]:
query = "Explain it to me like I am 5, what does Attention mean in an LLM ?"

result = adv_rag.query(
    question=query,
    top_k=3,
    min_score=0.1,
    summarize=True
)

print_rag_result(result)

Retrieving documents for query: 'Explain it to me like I am 5, what does Attention mean in an LLM ?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 11.80it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


QUESTION
Explain it to me like I am 5, what does Attention mean in an LLM ?

ANSWER
Imagine you're playing with your friends in a big room, and you want to tell them a story. But, there are lots of other kids talking and playing around you, and it's hard to focus on what you want to say.

"Attention" in a Large Language Model (LLM) is like a special helper that says, "Hey, I know you want to talk about something, but there's a lot of other stuff going on. Let me help you focus on the important parts."

It looks at what you're trying to say (that's the "query"), and then it looks at all the other things that might be related (those are the "key-value pairs"). Then, it says, "Oh, I see! This part is really important, and this part is not so important." And it helps you put all the important parts together to make your story make sense.

In the picture you showed me, it's like the model is looking at the word "making" and saying, "Oh, I see! This word is important, and it's related to the